In [2]:
#lưu file zip của nhánh feature/eda trên github về Drive để mở
from google.colab import drive
drive.mount('/content/drive')

!find /content/drive/MyDrive -name "stock-price-prediction-feature-eda.zip"

!cp "/content/drive/MyDrive/stock_price/stock-price-prediction-feature-eda.zip" ./data.zip
!unzip -q data.zip

print("Unzip thành công")

Mounted at /content/drive
/content/drive/MyDrive/stock_price/stock-price-prediction-feature-eda.zip
Unzip thành công


In [3]:
%cd /content/stock-price-prediction-feature-eda
!pwd

/content/stock-price-prediction-feature-eda
/content/stock-price-prediction-feature-eda


In [12]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
from src.data.crawl import config as cfg

#PATHS
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_STOCK_DIR = PROCESSED_DIR / "stock"
PROCESSED_MARKET_DIR = PROCESSED_DIR / "market"
PREPROCESSED_DIR=PROJECT_ROOT/"preprocess"

#COLUMNS
OHLCV_COLS = ["open", "high", "low", "close", "volume"]
QUALITY_COLS = [
    "missing_ohlc",
    "non_positive_ohlc",
    "high_invalid",
    "low_invalid",
    "ohlc_invalid",
    "zero_volume",
    "missing_volume"
]
EVENT_COLS = [
    "hose_disruption",
]
DERIVED_COLS = [
  "log_close","return","log_return","range_pct","traded_value","open_close_pct",
  "volatility_5d","volatility_20d", "realized_vol_20d","sma_5",
  "sma_20","ema_5","ema_20","close_vs_sma5","close_vs_sma20","momentum_5d", "momentum_20d",
  "atr_14_pct","volume_sma_20","rsi_14","lag_return_1","lag_return_5","atr_14",
  "excess_return_vnindex","excess_return_vn30","vnindex_log_return","vn30_log_return"]
HOSE_DISRUPTION_DATES = pd.to_datetime([
    "2018-01-23",
    "2018-01-24",
])

#RAW DATA + CLEANING DATA
def ensure_ohlcv_columns(df: pd.DataFrame) -> pd.DataFrame:
  #Các cột chuyển về dạng số: k có value-> NaN
    df = df.copy()
    for col in OHLCV_COLS:
        if col not in df.columns:
            df[col] = np.nan
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def load_raw_stock(code: str) -> pd.DataFrame:
    #Load raw stock CSV and normalize schema
    df = pd.read_csv(Path(cfg.RAW_STOCK_DIR)/ f"{code}.csv")
    df = df.rename(columns={"time": "trading_date"})
    df["trading_date"] = pd.to_datetime(df["trading_date"])
    df = ensure_ohlcv_columns(df)
    df = (
        df.sort_values("trading_date")
        .drop_duplicates("trading_date", keep="first")
        .reset_index(drop=True)
    )
    return df

def load_raw_market(symbol: str) -> pd.DataFrame:
    #Load raw market index CSV and normalize schema
    df = pd.read_csv(Path(cfg.RAW_MARKET_DIR) / f"{symbol}.csv")
    df = df.rename(columns={"time": "trading_date", "index": "index_code"})
    df["trading_date"] = pd.to_datetime(df["trading_date"])
    df = ensure_ohlcv_columns(df)
    df = (
        df.sort_values("trading_date")
        .drop_duplicates("trading_date", keep="first")
        .reset_index(drop=True)
    )
    return df

#QUALITY + EVENT FLAGS
def add_quality_flags(df: pd.DataFrame) -> pd.DataFrame:
  #flag: Missing value trong OHLC/<=0 value/'high' không cao nhất/'low' không nhỏ nhất
    df = df.copy()
    ohlc = ["open", "high", "low", "close"]
    df["missing_ohlc"] = df[ohlc].isna().any(axis=1).astype(int)
    df["non_positive_ohlc"] = (df[ohlc] <= 0).any(axis=1).astype(int)
    df["high_invalid"] = (
        (df["high"] < df["open"])
        | (df["high"] < df["close"])
        | (df["high"] < df["low"])
    ).astype(int)
    df["low_invalid"] = (
        (df["low"] > df["open"])
        | (df["low"] > df["close"])
        | (df["low"] > df["high"])
    ).astype(int)
    df["ohlc_invalid"] = (
        df["missing_ohlc"].eq(1)
        | df["non_positive_ohlc"].eq(1)
        | df["high_invalid"].eq(1)
        | df["low_invalid"].eq(1)
    ).astype(int)
    if "volume" in df.columns:
        df["zero_volume"] = df["volume"].eq(0).astype(int)
        df["missing_volume"] = df["volume"].isna().astype(int)
    else:
        df["zero_volume"] = 0
        df["missing_volume"] = 0
    return df

def add_event_flags(df: pd.DataFrame) -> pd.DataFrame:
  #event flag: các sự kiện gián đoạn giao dịch
  #HOSE disruption: 23-24/1/2018
    df = df.copy()
    df["hose_disruption"] = (
        df["trading_date"].isin(HOSE_DISRUPTION_DATES)
    ).astype(int)
    return df

#FEATURES
#một vài features cho model học
def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = add_quality_flags(df)
    df = add_event_flags(df)

    bad_data_mask = (df["ohlc_invalid"].eq(1)) | (df["hose_disruption"].eq(1)) | (df["zero_volume"].eq(1))| df["missing_volume"].eq(1)

    df_clean_time = df.copy()
    df_clean_time.loc[bad_data_mask, OHLCV_COLS] = np.nan
    df_clean_time[OHLCV_COLS[:4]] = df_clean_time[OHLCV_COLS[:4]].ffill()
    #Close-based
    valid_close = df_clean_time["close"].gt(0) & df_clean_time["close"].notna()
    df["log_close"] = np.where(valid_close, np.log(df_clean_time["close"]), np.nan)
    df["return"] = df_clean_time["close"].pct_change()
    df["log_return"] = pd.Series(df["log_close"], index=df.index).diff()
    #OHLC based
    clean_ohlc = df[["open", "high", "low", "close"]].mask(bad_data_mask)
    df["range_pct"] = (clean_ohlc["high"] - clean_ohlc["low"]) / clean_ohlc["close"]
    df["open_close_pct"] = (clean_ohlc["close"] - clean_ohlc["open"]) / clean_ohlc["open"]
    #votality+ATR
    df["volatility_5d"] = (df["log_return"].rolling(5).std())
    df["volatility_20d"] = (df["log_return"].rolling(20).std())
    df["realized_vol_20d"] = np.sqrt((df["log_return"] ** 2).rolling(20).sum())

    high_low = df_clean_time["high"] - df_clean_time["low"]
    high_close_prev = (df_clean_time["high"] - df_clean_time["close"].shift(1)).abs()
    low_close_prev = (df_clean_time["low"] - df_clean_time["close"].shift(1)).abs()
    df["true_range"] = pd.concat([high_low, high_close_prev, low_close_prev], axis=1).max(axis=1)

    window = 14
    df["atr_14"] = df["true_range"].ewm(alpha=1/window, adjust=False).mean()
    df["atr_14_pct"] = df["atr_14"] / df_clean_time["close"]
    df.drop(columns=["true_range"], inplace=True)

    # Trend + Momentum
    df["sma_5"] = df_clean_time["close"].rolling(5).mean()
    df["sma_20"] = df_clean_time["close"].rolling(20).mean()
    df["ema_5"] = df_clean_time["close"].ewm(span=5, adjust=False).mean()
    df["ema_20"] = df_clean_time["close"].ewm(span=20, adjust=False).mean()
    df["close_vs_sma5"] = (df_clean_time["close"] / df["sma_5"]) - 1
    df["close_vs_sma20"] = (df_clean_time["close"] / df["sma_20"]) - 1
    df["momentum_5d"] = (df_clean_time["close"] / df_clean_time["close"].shift(5)) - 1
    df["momentum_20d"] = (df_clean_time["close"] / df_clean_time["close"].shift(20)) - 1

    # Volume-based feature
    if "volume" in df.columns:
        df["traded_value"] = df_clean_time["close"] * df_clean_time["volume"]
        df["volume_sma_20"] = df_clean_time["volume"].rolling(20).mean()

    # RSI
    delta = df_clean_time["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/window, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    df["rsi_14"] = 100 - (100 / (1 + rs))

    # Lag
    df["lag_return_1"] = df["log_return"].shift(1)
    df["lag_return_5"] = df["log_return"].shift(5)

    return df

def add_market_related_features(stock_df: pd.DataFrame,vnindex_df: pd.DataFrame,vn30_df: pd.DataFrame) -> pd.DataFrame:

    df = stock_df.copy()
    vnindex = (vnindex_df[["trading_date", "log_return"]].rename(columns={"log_return": "vnindex_log_return"}))
    vn30 = (vn30_df[["trading_date", "log_return"]].rename(columns={"log_return": "vn30_log_return"}))
    df = df.merge(vnindex,on="trading_date", how="left")
    df = df.merge(vn30,on="trading_date",how="left")
    # excess return so với toàn thị trường
    df["excess_return_vnindex"] = (df["log_return"]-df["vnindex_log_return"])
    # excess return so với nhóm bluechip
    df["excess_return_vn30"] = (df["log_return"]-df["vn30_log_return"])
    return df

#TARGET COLUMN
#log_return mới là thứ mà model dự đoán chứ không phải price thuần (log_return có thể xem đại khái như tỉ lệ giá tăng)
def add_target_columns(df:pd.DataFrame) -> pd.DataFrame:
  df=df.copy()

  df["target_log_return_1d"] = df["log_return"].shift(-1) #D1 biết độ tăng D0->D1 là 1% thì D2 predict độ tăng từ D1->D2
  df["target_log_return_5d"] = (df["log_return"].rolling(5).sum().shift(-5)) #nghĩa là giá tăng bao nhiêu sau 5 ngày
  df["target_log_return_20d"] = (df["log_return"].rolling(20).sum().shift(-20)) #sau 20 ngày

  return df

#HANDLE FLAGS
def handle_flag(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    flag_days = ((df["ohlc_invalid"] == 1) | (df["hose_disruption"] == 1) | (df["zero_volume"] == 1) |(df["missing_volume"] == 1))
    df = df[~flag_days]
    essential_features = DERIVED_COLS
    df = df.dropna(subset=essential_features)
    target_cols = ["target_log_return_1d", "target_log_return_5d", "target_log_return_20d"]
    df = df.dropna(subset=target_cols)
    return df.reset_index(drop=True)

def preprocess(stock_code: str, vnindex_market_df: pd.DataFrame,vn30_market_df) -> pd.DataFrame:
    df = load_raw_stock(stock_code)
    df = add_derived_columns(df)
    df = add_market_related_features(df, vnindex_market_df,vn30_market_df)
    df = add_target_columns(df)
    df_clean = handle_flag(df)
    return df_clean

#MAIN
def main() -> None:
    PROCESSED_STOCK_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSED_MARKET_DIR.mkdir(parents=True, exist_ok=True)
    PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    vnindex_market_df = load_raw_market("VNINDEX")
    vnindex_market_df = add_derived_columns(vnindex_market_df)
    vnindex_market_df.to_csv(PROCESSED_MARKET_DIR / "VNINDEX_processed.csv", index=False)

    vn30_market_df = load_raw_market("VN30")
    vn30_market_df = add_derived_columns(vn30_market_df)
    vn30_market_df.to_csv(PROCESSED_MARKET_DIR / "VN30_processed.csv", index=False)

    sample_stocks = ["FPT","HPG","VCB","VIC","VNM"]

    processed_df=dict()
    for code in sample_stocks:
        processed_df[code] = preprocess(code, vnindex_market_df, vn30_market_df)

    #Nếu train riêng trên từng mã thỉ bỏ từ đoạn này tới dòng comment tiếp theo
    common_dates = set(processed_df[sample_stocks[0]]["trading_date"])
    for code in sample_stocks[1:]:
        common_dates &= set(processed_df[code]["trading_date"])

    common_dates = sorted(common_dates)

    for code, df in processed_df.items():
        df = df[df["trading_date"].isin(common_dates)]
        #bỏ tới đây
        output_file = PREPROCESSED_DIR / f"{code}_preprocessed.csv"
        df.to_csv(output_file, index=False)
        print(f"Đã xuất file: {output_file}")

if __name__ == "__main__":
    main()

Đã xuất file: /content/stock-price-prediction-feature-eda/preprocess/FPT_preprocessed.csv
Đã xuất file: /content/stock-price-prediction-feature-eda/preprocess/HPG_preprocessed.csv
Đã xuất file: /content/stock-price-prediction-feature-eda/preprocess/VCB_preprocessed.csv
Đã xuất file: /content/stock-price-prediction-feature-eda/preprocess/VIC_preprocessed.csv
Đã xuất file: /content/stock-price-prediction-feature-eda/preprocess/VNM_preprocessed.csv


In [13]:
tickers = ["FPT","HPG","VCB","VIC","VNM"]
dfs = []

for ticker in tickers:
    file_path = PREPROCESSED_DIR / f"{ticker}_preprocessed.csv"
    df = pd.read_csv(file_path)
    df["trading_date"] = pd.to_datetime(df["trading_date"])
    df["ticker"] = ticker
    dfs.append(df)

all_df = pd.concat(dfs,ignore_index=True)
all_df = (all_df.sort_values(["trading_date", "ticker"]).reset_index(drop=True))

print(all_df.shape)
print(all_df.head())

(17155, 46)
  stock_code trading_date   open   high    low  close  volume  missing_ohlc  \
0        FPT   2012-02-07   4.23   4.27   4.18   4.23   74910             0   
1        HPG   2012-02-07   0.80   0.81   0.79   0.80  563860             0   
2        VCB   2012-02-07   5.55   5.62   5.44   5.55  502100             0   
3        VIC   2012-02-07   9.84  10.01   9.59   9.76   74480             0   
4        VNM   2012-02-07  12.72  12.72  12.64  12.72   52570             0   

   non_positive_ohlc  high_invalid  ...  lag_return_1  lag_return_5  \
0                  0             0  ...      0.000000     -0.004796   
1                  0             0  ...      0.038221      0.000000   
2                  0             0  ...     -0.007233      0.009083   
3                  0             0  ...      0.034112      0.008840   
4                  0             0  ...     -0.005488      0.012005   

   vnindex_log_return  vn30_log_return  excess_return_vnindex  \
0            0.003372

In [14]:
#Time series split/forward chaining
from sklearn.model_selection import TimeSeriesSplit
dates = np.sort(all_df["trading_date"].unique())

target='target_log_return_20d'
tscv = TimeSeriesSplit(n_splits=5,gap=20) #gap=20/21 để tránh leakage nhé, có thể đổi theo target

for fold, (train_date_idx, test_date_idx) in enumerate(tscv.split(dates)):
    train_dates = dates[train_date_idx]
    test_dates = dates[test_date_idx]
    train_df = all_df[all_df["trading_date"].isin(train_dates)]
    test_df = all_df[all_df["trading_date"].isin(test_dates)]
    print(f"Fold {fold+1}: "f"train={len(train_df)}, "f"test={len(test_df)}")

Fold 1: train=2780, test=2855
Fold 2: train=5635, test=2855
Fold 3: train=8490, test=2855
Fold 4: train=11345, test=2855
Fold 5: train=14200, test=2855


In [ ]:
#Cho SVR
#trong fold nên thêm scaler: RobustScaler
#thử thêm nhiều loại scaler nữa nhé
#Cho prophet + ARIMA
#train riêng trên từng csv của các mã, không dùng all_df

#sau khi train xong mn có thể xem feature selection
